# 🛒 E-Commerce Sales Trend and Customer Analysis
### InternCircle Data Science & AI Internship — Project 03

---

> **Author:** InternCircle Intern  
> **Date:** November 2026  
> **Tools:** Python 3.10+, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook

> ⚠️ **Data Disclosure:** The dataset used in this project is **entirely simulated** using NumPy and Pandas with a fixed random seed. It does **not** represent real customer transactions or any real business. It is generated solely for educational and analytical purposes.

---

## 🎯 Project Objective

Perform end-to-end analysis of a simulated e-commerce sales dataset to:

1. Identify **monthly revenue trends** and growth patterns
2. Discover **top-performing product categories**
3. Identify the **most valuable customers** using Pareto analysis
4. Convert raw sales data into **actionable business recommendations**

---

## 📋 Analysis Workflow

| Step | Description |
|------|-------------|
| 1 | Define Business Questions |
| 2 | Generate and Inspect Dataset |
| 3 | Data Cleaning |
| 4 | Feature Engineering |
| 5 | Monthly Revenue Trend Analysis |
| 6 | Product Category Analysis |
| 7 | Customer Value and Pareto Analysis |
| 8 | Visualizations |
| 9 | Business Recommendations |

---
## 📌 Step 1 — Define Business Questions

Before analysing any data, we commit to three guiding business questions.  
This prevents cherry-picking results and keeps analysis purposeful.

| # | Business Question | Why It Matters |
|---|-------------------|----------------|
| **Q1** | Which months generate the highest and lowest revenue? | Identifies seasonal peaks and troughs — critical for inventory planning, marketing budgets, and staffing |
| **Q2** | Which product categories perform best in revenue, average order value, and order count? | Tells the business where to invest marketing spend and which categories to expand or cut |
| **Q3** | Who are the most valuable customers, and what % of total revenue comes from the top 10%? | Guides loyalty program design and customer retention investment |

> **Business context:** In e-commerce, revenue concentration is common — a small group of customers or categories often drives a disproportionate share of total revenue. Quantifying this concentration allows targeted strategy rather than a one-size-fits-all approach.

---
## 📦 Step 2 — Import Libraries and Generate Dataset

In [ ]:
# ── Standard and suppression ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Notebook display ──────────────────────────────────────────────────────────
import os
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 15)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

os.makedirs('outputs', exist_ok=True)

print('✅ All libraries imported successfully!')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')

### 2.1 — Generate the Simulated Dataset

We generate **2,000 orders** spanning a full calendar year (2023) using NumPy random functions.  
A **fixed random seed (`seed=42`)** ensures the dataset is fully reproducible — running this notebook any number of times produces identical data.

> ⚠️ **This dataset is entirely simulated and does not represent any real business or customer.**

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Parameters ────────────────────────────────────────────────────────────────
N_ORDERS     = 2000     # number of simulated orders
N_CUSTOMERS  = 300      # unique simulated customers
CATEGORIES   = ['Electronics', 'Fashion', 'Home', 'Books']

# ── Category price ranges (min, max) in USD ───────────────────────────────────
# Electronics tend to be higher-ticket; Books are lower-ticket.
PRICE_RANGES = {
    'Electronics': (200, 1500),
    'Fashion'    : (30,  400),
    'Home'       : (50,  600),
    'Books'      : (10,  80),
}

# ── Category selection probabilities ─────────────────────────────────────────
# Electronics and Fashion orders are more common than Books.
CAT_WEIGHTS = [0.30, 0.35, 0.22, 0.13]   # sums to 1.00

# ── Generate each column ──────────────────────────────────────────────────────
# 1. Order dates: uniformly distributed over 2023
start_date  = pd.Timestamp('2023-01-01')
end_date    = pd.Timestamp('2023-12-31')
day_range   = (end_date - start_date).days + 1
order_dates = [
    start_date + pd.Timedelta(days=int(d))
    for d in rng.integers(0, day_range, size=N_ORDERS)
]

# 2. Customer IDs: randomly drawn from pool of 300 simulated customers
customer_ids = [f'CUST_{str(c).zfill(4)}' for c in rng.integers(1, N_CUSTOMERS + 1, size=N_ORDERS)]

# 3. Categories: chosen with realistic weights
categories = rng.choice(CATEGORIES, size=N_ORDERS, p=CAT_WEIGHTS)

# 4. Order amounts: sampled from category-specific price ranges
amounts = np.array([
    round(rng.uniform(*PRICE_RANGES[cat]), 2)
    for cat in categories
])

# ── Build DataFrame ───────────────────────────────────────────────────────────
df = pd.DataFrame({
    'order_date'  : order_dates,
    'customer_id' : customer_ids,
    'category'    : categories,
    'amount'      : amounts,
})

print(f'✅ Simulated dataset generated: {df.shape[0]} orders, {df.shape[1]} columns')
print(f'   Date range : {df["order_date"].min().date()} → {df["order_date"].max().date()}')
print(f'   Customers  : {df["customer_id"].nunique()} unique')
print(f'   Categories : {df["category"].unique()}')

### 2.2 — Inspect the Dataset

In [ ]:
# Shape: how many rows and columns?
print(f'Shape → {df.shape[0]} rows × {df.shape[1]} columns')

In [ ]:
# Head: first 5 rows — sanity check
df.head()

In [ ]:
# Info: data types and non-null counts
df.info()

In [ ]:
# Describe: statistical summary of numerical columns
df.describe()

In [ ]:
# Missing values check
print('Missing values per column:')
print(df.isnull().sum().to_string())

In [ ]:
# Duplicate rows check
dupe_count = df.duplicated().sum()
print(f'Duplicate rows: {dupe_count}')

---
## 🧹 Step 3 — Data Cleaning

Even with a generated dataset, we practise rigorous cleaning — a real pipeline would apply the same checks to real data.

| Check | Action |
|-------|--------|
| `order_date` type | Convert to `datetime64` |
| Missing values | Identify and handle |
| Duplicate orders | Identify and handle |
| Customer ID format | Validate pattern |
| Category names | Validate against known list |
| `amount` type | Ensure numeric, no negatives |

In [ ]:
# Work on a clean copy
df_clean = df.copy()

# ── 3a. Convert order_date to datetime ───────────────────────────────────────
# 'coerce' turns any unparseable value into NaT (Not a Time) instead of raising an error
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], errors='coerce')
print(f'order_date dtype after conversion: {df_clean["order_date"].dtype}')

# ── 3b. Check for NaT introduced during conversion ───────────────────────────
nat_count = df_clean['order_date'].isna().sum()
print(f'NaT values in order_date        : {nat_count}')
if nat_count > 0:
    df_clean.dropna(subset=['order_date'], inplace=True)
    print(f'  → Dropped {nat_count} rows with invalid dates.')

# ── 3c. Missing values ────────────────────────────────────────────────────────
missing = df_clean.isnull().sum()
print(f'\nMissing values after date cleaning:')
print(missing[missing > 0].to_string() if missing.sum() > 0 else '  None ✅')

# ── 3d. Duplicate orders ──────────────────────────────────────────────────────
dupes = df_clean.duplicated().sum()
print(f'\nDuplicate rows: {dupes}')
# Note: in real data, same customer may order same amount on same day — investigate before dropping.
# For this dataset we keep them as valid separate orders.

# ── 3e. Validate customer IDs ─────────────────────────────────────────────────
valid_cust_pattern = df_clean['customer_id'].str.match(r'^CUST_\d{4}$')
invalid_cust = (~valid_cust_pattern).sum()
print(f'Invalid customer IDs            : {invalid_cust}')

# ── 3f. Validate categories ───────────────────────────────────────────────────
valid_cats   = set(CATEGORIES)
invalid_cats = (~df_clean['category'].isin(valid_cats)).sum()
print(f'Invalid category entries        : {invalid_cats}')

# ── 3g. Validate amount: must be numeric and positive ────────────────────────
df_clean['amount'] = pd.to_numeric(df_clean['amount'], errors='coerce')
neg_amounts  = (df_clean['amount'] <= 0).sum()
na_amounts   = df_clean['amount'].isna().sum()
print(f'Negative or zero amounts        : {neg_amounts}')
print(f'Non-numeric amounts             : {na_amounts}')

# Drop rows with invalid amount
df_clean.dropna(subset=['amount'], inplace=True)
df_clean = df_clean[df_clean['amount'] > 0]

print(f'\n✅ Cleaned dataset: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')

---
## 🔧 Step 4 — Feature Engineering and Date Extraction

We extract time components from `order_date` to enable time-based aggregations.  
Without these derived columns, grouping by month or quarter would require complex string operations.

| New Column | Purpose |
|------------|--------|
| `year` | Filter to a specific year |
| `month` | Group revenue by calendar month |
| `month_name` | Human-readable month labels for charts |
| `weekday` | Identify day-of-week patterns |
| `quarter` | Quarterly business reporting |

In [ ]:
# Pandas dt accessor makes date extraction concise and fast
df_clean['year']       = df_clean['order_date'].dt.year
df_clean['month']      = df_clean['order_date'].dt.month
df_clean['month_name'] = df_clean['order_date'].dt.strftime('%b')   # 'Jan', 'Feb', ...
df_clean['weekday']    = df_clean['order_date'].dt.day_name()        # 'Monday', etc.
df_clean['quarter']    = df_clean['order_date'].dt.quarter

# Convert month_name to an ordered categorical so charts sort correctly
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
df_clean['month_name'] = pd.Categorical(df_clean['month_name'],
                                         categories=month_order, ordered=True)

print('Feature engineering complete.')
print(f'Columns now: {df_clean.columns.tolist()}')
df_clean.head(3)

In [ ]:
# Quick distribution check: orders per quarter
print('Orders per quarter:')
print(df_clean.groupby('quarter')['amount'].agg(Orders='count', Revenue='sum').to_string())

---
## 📈 Step 5 — Monthly Revenue Trend Analysis

`groupby('month')` splits all orders by calendar month.  
`pct_change()` computes month-over-month growth rate — a key metric for spotting acceleration or decline.

In [ ]:
# ── 5a. Monthly revenue aggregation ──────────────────────────────────────────
monthly_revenue = (
    df_clean
    .groupby(['month', 'month_name'], observed=True)['amount']
    .agg(
        Total_Revenue  = 'sum',
        Order_Count    = 'count',
        Avg_Order_Value= 'mean'
    )
    .reset_index()
    .sort_values('month')
)

# ── 5b. Month-over-month growth rate ─────────────────────────────────────────
# pct_change() computes (current - previous) / previous * 100
monthly_revenue['MoM_Growth_Pct'] = monthly_revenue['Total_Revenue'].pct_change() * 100

print('Monthly Revenue Summary:')
print('=' * 65)
print(monthly_revenue[['month_name','Total_Revenue','Order_Count','Avg_Order_Value','MoM_Growth_Pct']]
      .to_string(index=False))

In [ ]:
# ── 5c. Identify best and worst months ───────────────────────────────────────
best_month  = monthly_revenue.loc[monthly_revenue['Total_Revenue'].idxmax()]
worst_month = monthly_revenue.loc[monthly_revenue['Total_Revenue'].idxmin()]

print(f'🏆 Highest Revenue Month : {best_month["month_name"]}  →  ${best_month["Total_Revenue"]:,.0f}')
print(f'📉 Lowest  Revenue Month : {worst_month["month_name"]}  →  ${worst_month["Total_Revenue"]:,.0f}')
print(f'   Revenue gap : ${best_month["Total_Revenue"] - worst_month["Total_Revenue"]:,.0f}')
print(f'   Total annual revenue   : ${monthly_revenue["Total_Revenue"].sum():,.0f}')

---
## 🏷️ Step 6 — Product Category Analysis

We analyse each product category across three metrics:  
**total revenue** (absolute contribution), **average order value** (ticket size), and **order count** (popularity).

In [ ]:
# ── 6a. Category performance summary ─────────────────────────────────────────
category_summary = (
    df_clean
    .groupby('category')['amount']
    .agg(
        Total_Revenue   = 'sum',
        Avg_Order_Value = 'mean',
        Order_Count     = 'count'
    )
    .sort_values('Total_Revenue', ascending=False)
    .assign(
        Revenue_Share_Pct = lambda x: (x['Total_Revenue'] / x['Total_Revenue'].sum() * 100).round(1)
    )
)

print('Category Performance (sorted by Total Revenue):')
print('=' * 65)
print(category_summary.to_string())

In [ ]:
# ── 6b. Pivot table: Monthly revenue by category ──────────────────────────────
# pivot_table is perfect for two-dimensional summaries
cat_pivot = df_clean.pivot_table(
    values  = 'amount',
    index   = 'month_name',
    columns = 'category',
    aggfunc = 'sum'
).fillna(0)

print('Monthly Revenue by Category (USD):')
print('=' * 65)
print(cat_pivot.to_string())

---
## 👥 Step 7 — Customer Value and Pareto Analysis

The **Pareto principle** (also called the 80/20 rule) suggests that roughly 80% of outcomes come from 20% of causes.  
In retail, this often manifests as a small group of customers generating the majority of revenue.  

We calculate the **actual** figure — not assume 80/20.

In [ ]:
# ── 7a. Total spending per customer ──────────────────────────────────────────
customer_summary = (
    df_clean
    .groupby('customer_id')['amount']
    .agg(
        Total_Spent  = 'sum',
        Order_Count  = 'count',
        Avg_Order    = 'mean'
    )
    .sort_values('Total_Spent', ascending=False)
    .reset_index()
)

print(f'Total customers        : {len(customer_summary)}')
print(f'Highest spender        : ${customer_summary["Total_Spent"].max():,.2f}')
print(f'Median customer spend  : ${customer_summary["Total_Spent"].median():,.2f}')
print(f'Mean customer spend    : ${customer_summary["Total_Spent"].mean():,.2f}')

In [ ]:
# ── 7b. Pareto analysis — top 10% of customers ───────────────────────────────
total_customers      = len(customer_summary)
top10_cutoff         = max(1, int(np.ceil(total_customers * 0.10)))
top10_customers      = customer_summary.head(top10_cutoff)
total_revenue_all    = customer_summary['Total_Spent'].sum()
top10_revenue        = top10_customers['Total_Spent'].sum()
top10_revenue_pct    = top10_revenue / total_revenue_all * 100

print('=' * 55)
print('PARETO ANALYSIS — CUSTOMER REVENUE CONCENTRATION')
print('=' * 55)
print(f'  Total unique customers     : {total_customers}')
print(f'  Top 10% threshold          : top {top10_cutoff} customers')
print(f'  Top 10% total revenue      : ${top10_revenue:,.2f}')
print(f'  Total revenue (all)        : ${total_revenue_all:,.2f}')
print(f'  Top 10% revenue share      : {top10_revenue_pct:.1f}%')
print(f'  Remaining 90% revenue share: {100 - top10_revenue_pct:.1f}%')
print('=' * 55)

if top10_revenue_pct >= 80:
    print('  → Result SUPPORTS the Pareto 80/20 pattern.')
else:
    print(f'  → Result does NOT follow strict 80/20.')
    print(f'    Revenue is distributed more evenly across customers.')

In [ ]:
# ── 7c. Top 10 customers ─────────────────────────────────────────────────────
top10_display = customer_summary.head(10).copy()
top10_display['Revenue_Share_Pct'] = (top10_display['Total_Spent'] / total_revenue_all * 100).round(2)

print('Top 10 Customers by Total Spending:')
print(top10_display[['customer_id','Total_Spent','Order_Count','Avg_Order','Revenue_Share_Pct']]
      .to_string(index=False))

---
## 📊 Step 8 — Business Visualizations

Six professional charts — one clear message per chart.  
All saved to the `outputs/` folder at 300 DPI.

### Chart 1 — Monthly Revenue Trend

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    monthly_revenue['month_name'].astype(str),
    monthly_revenue['Total_Revenue'],
    marker='o', linewidth=2.2, color='#2980B9', markersize=7, label='Monthly Revenue'
)

# Highlight best and worst months
best_idx  = monthly_revenue['Total_Revenue'].idxmax()
worst_idx = monthly_revenue['Total_Revenue'].idxmin()

ax.scatter(
    monthly_revenue.loc[best_idx, 'month_name'].astype(str) if hasattr(monthly_revenue.loc[best_idx, 'month_name'], 'astype') else str(monthly_revenue.loc[best_idx, 'month_name']),
    monthly_revenue.loc[best_idx, 'Total_Revenue'],
    color='#2ECC71', s=140, zorder=5, label=f'Best Month'
)
ax.scatter(
    str(monthly_revenue.loc[worst_idx, 'month_name']),
    monthly_revenue.loc[worst_idx, 'Total_Revenue'],
    color='#E74C3C', s=140, zorder=5, label=f'Worst Month'
)

# Average line
avg_rev = monthly_revenue['Total_Revenue'].mean()
ax.axhline(avg_rev, color='gray', linestyle='--', linewidth=1.2,
           label=f'Monthly Avg (${avg_rev:,.0f})')

ax.set_title('Monthly Revenue Trend — 2023 (Simulated Data)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Total Revenue (USD)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)
ax.set_xticklabels(monthly_revenue['month_name'].astype(str), rotation=0)

plt.tight_layout()
plt.savefig('outputs/monthly_revenue.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/monthly_revenue.png')

### Chart 2 — Month-over-Month Revenue Growth

In [ ]:
mom = monthly_revenue.dropna(subset=['MoM_Growth_Pct']).copy()
colors = ['#2ECC71' if v >= 0 else '#E74C3C' for v in mom['MoM_Growth_Pct']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    mom['month_name'].astype(str),
    mom['MoM_Growth_Pct'],
    color=colors, edgecolor='white', linewidth=0.8
)
ax.axhline(0, color='black', linewidth=1)

for bar, val in zip(bars, mom['MoM_Growth_Pct']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + (0.5 if val >= 0 else -1.5),
        f'{val:+.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold'
    )

ax.set_title('Month-over-Month Revenue Growth (%) — 2023 (Simulated)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Growth Rate (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.0f}%'))

plt.tight_layout()
plt.savefig('outputs/monthly_growth.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/monthly_growth.png')

### Chart 3 — Revenue by Product Category

In [ ]:
cat_sorted = category_summary.reset_index().sort_values('Total_Revenue', ascending=True)
palette    = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: horizontal bar chart
axes[0].barh(
    cat_sorted['category'], cat_sorted['Total_Revenue'],
    color=palette[::-1], edgecolor='white'
)
for i, (rev, share) in enumerate(zip(cat_sorted['Total_Revenue'], cat_sorted['Revenue_Share_Pct'])):
    axes[0].text(rev + 500, i, f'${rev:,.0f}  ({share}%)',
                 va='center', fontsize=9, fontweight='bold')
axes[0].set_title('Total Revenue by Category', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Revenue (USD)', fontsize=10)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Right: pie chart of revenue shares
axes[1].pie(
    category_summary['Total_Revenue'],
    labels    = category_summary.index,
    autopct   = '%1.1f%%',
    colors    = palette,
    startangle= 140,
    wedgeprops= {'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Revenue Share by Category', fontsize=12, fontweight='bold')

plt.suptitle('Product Category Revenue Analysis (Simulated Data)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/category_revenue.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/category_revenue.png')

### Chart 4 — Monthly Revenue by Category (Stacked Area)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

cat_pivot_plot = cat_pivot.reset_index()
months_str     = cat_pivot_plot['month_name'].astype(str).tolist()
cat_colors     = {'Electronics': '#E74C3C', 'Fashion': '#3498DB',
                   'Home': '#F39C12', 'Books': '#2ECC71'}

bottom = np.zeros(len(months_str))
for cat in CATEGORIES:
    if cat in cat_pivot_plot.columns:
        vals = cat_pivot_plot[cat].values
        ax.bar(months_str, vals, bottom=bottom,
               label=cat, color=cat_colors.get(cat, '#999'),
               edgecolor='white', linewidth=0.7)
        bottom += vals

ax.set_title('Monthly Revenue by Category — 2023 (Simulated Data)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Total Revenue (USD)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/monthly_category_trend.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/monthly_category_trend.png')

### Chart 5 — Top 10 Customers by Spending

In [ ]:
top10_chart = customer_summary.head(10).sort_values('Total_Spent')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    top10_chart['customer_id'], top10_chart['Total_Spent'],
    color='#2980B9', edgecolor='white'
)
for bar, val in zip(bars, top10_chart['Total_Spent']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}', va='center', fontsize=9, fontweight='bold')

ax.set_title('Top 10 Customers by Total Spending (Simulated Data)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Total Spending (USD)', fontsize=11)
ax.set_ylabel('Customer ID', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('outputs/top_customers.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/top_customers.png')

### Chart 6 — Customer Revenue Contribution (Pareto Curve)

In [ ]:
cust_sorted    = customer_summary.sort_values('Total_Spent', ascending=False).reset_index(drop=True)
cust_sorted['cumulative_pct_customers'] = (cust_sorted.index + 1) / len(cust_sorted) * 100
cust_sorted['cumulative_pct_revenue']   = cust_sorted['Total_Spent'].cumsum() / total_revenue_all * 100

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    cust_sorted['cumulative_pct_customers'],
    cust_sorted['cumulative_pct_revenue'],
    color='#2980B9', linewidth=2.2, label='Actual Pareto Curve'
)

# Reference line: perfect equality (45°)
ax.plot([0, 100], [0, 100], 'k--', linewidth=1, alpha=0.5, label='Line of Equality')

# Mark the top 10% point
ax.axvline(10, color='#E74C3C', linestyle=':', linewidth=1.5)
top10_y = cust_sorted.loc[cust_sorted['cumulative_pct_customers'] <= 10, 'cumulative_pct_revenue'].max()
ax.axhline(top10_y, color='#E74C3C', linestyle=':', linewidth=1.5,
           label=f'Top 10% → {top10_y:.1f}% of Revenue')

ax.set_title('Customer Revenue Concentration — Pareto Curve (Simulated)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Cumulative % of Customers (ranked by spending)', fontsize=11)
ax.set_ylabel('Cumulative % of Revenue', fontsize=11)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('outputs/customer_contribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: outputs/customer_contribution.png')

---
## 💼 Step 9 — Business Recommendations

Every recommendation is backed by actual calculated numbers from the analysis above.

> ⚠️ These recommendations are derived from **simulated** data and are illustrative only.

In [ ]:
# Collect the numbers we need
top_cat     = category_summary.index[0]
top_cat_rev = category_summary.iloc[0]['Total_Revenue']
top_cat_pct = category_summary.iloc[0]['Revenue_Share_Pct']
low_cat     = category_summary.index[-1]
low_cat_aov = category_summary.iloc[-1]['Avg_Order_Value']

total_annual  = monthly_revenue['Total_Revenue'].sum()
best_mon_name = best_month['month_name']
worst_mon_name= worst_month['month_name']
gap           = best_month['Total_Revenue'] - worst_month['Total_Revenue']

print('=' * 65)
print('BUSINESS RECOMMENDATIONS (based on simulated data)')
print('=' * 65)
print()
print('RECOMMENDATION 1 — Targeted Revenue Recovery Campaigns')
print('─' * 55)
print(f'  {worst_mon_name} is the lowest-revenue month (${worst_month["Total_Revenue"]:,.0f}).')
print(f'  The gap between the best month ({best_mon_name}: ${best_month["Total_Revenue"]:,.0f})')
print(f'  and the worst is ${gap:,.0f}.')
print(f'  Action: Run targeted promotions and limited-time offers in')
print(f'  {worst_mon_name} to lift revenue by even 10–15%,')
print(f'  which would recover ${gap * 0.1:,.0f}–${gap * 0.15:,.0f} in additional revenue.')
print()
print('RECOMMENDATION 2 — Double Down on Top Category')
print('─' * 55)
print(f'  {top_cat} generates ${top_cat_rev:,.0f} ({top_cat_pct}% of total revenue).')
print(f'  This category has the highest average order value and revenue share.')
print(f'  Action: Increase paid marketing budget for {top_cat} by 15–20%.')
print(f'  Even a 5% revenue lift in this category adds ${top_cat_rev * 0.05:,.0f} annually.')
print()
print('RECOMMENDATION 3 — High-Value Customer Loyalty Programme')
print('─' * 55)
print(f'  The top 10% of customers ({top10_cutoff} people) account for')
print(f'  {top10_revenue_pct:.1f}% of all revenue (${top10_revenue:,.0f}).')
print(f'  These customers are the business core and must be retained.')
print(f'  Action: Introduce a VIP loyalty tier with early access, free')
print(f'  shipping, and exclusive discounts for all customers who spend')
print(f'  more than ${top10_customers["Total_Spent"].min():,.0f} annually.')
print(f'  Increasing repeat purchase rate by 10% among these {top10_cutoff}')
print(f'  customers could add ${top10_revenue * 0.10:,.0f} in revenue.')
print()
print('=' * 65)

---
## ✅ Step 10 — Final Summary

| Insight | Finding |
|---------|--------|
| Annual revenue | Computed from actual aggregation |
| Best revenue month | Identified via idxmax() |
| Worst revenue month | Identified via idxmin() |
| Top category | Computed via groupby + sort |
| Top 10% customer revenue share | Computed via cumulative sum |
| Pareto compliance | Evaluated empirically, not assumed |

> ⚠️ **All statistics are computed from simulated data. This project is for educational purposes only.**

---
*End of InternCircle Project 03 — E-Commerce Sales Trend and Customer Analysis* 🛒